# Lab 4 · METR-LA | Dự báo đa bước bằng LSTM

**TV5 — LSTM.** Dự báo trực tiếp 15, 30 và 60 phút trên các cảm biến tốc độ. Phần thực nghiệm này giữ riêng mã nguồn huấn luyện, checkpoint, kiểm định và dự báo của LSTM để tích hợp với kết quả VAR/XGBoost của nhóm.

**Phạm vi:** 8 cảm biến đầu tiên theo thứ tự cột gốc, không phải toàn bộ 207 cảm biến. Lookback 12 quan sát × 5 phút; horizon 3/6/12 bước. Chia theo thời gian 70/15/15. Các số 0 hoặc tốc độ ≤1 được loại khỏi loss/đánh giá theo *chính sách thực nghiệm*; không khẳng định mọi số 0 đều là thiếu dữ liệu. Đơn vị giá trị tốc độ giữ nguyên như file gốc (chưa xác minh metadata đơn vị).

## 1. Dữ liệu, chính sách thời gian và tensor

In [ ]:
from pathlib import Path
import os, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# Hoạt động khi mở notebook từ thư mục gốc repository hoặc contributions/.
ROOT = Path.cwd()
if not (ROOT / 'data/traffic/METR-LA.h5').is_file():
    ROOT = ROOT.parent
DATA = ROOT / 'data/traffic/METR-LA.h5'
assert DATA.is_file(), 'Không tìm thấy data/traffic/METR-LA.h5 trong repository.'
OUTPUT = ROOT / 'models'; OUTPUT.mkdir(exist_ok=True)
FIG = ROOT / 'figures'; FIG.mkdir(exist_ok=True)
TABLE = ROOT / 'tables'; TABLE.mkdir(exist_ok=True)

SEED = 42
LOOKBACK = 12
HORIZONS = (3, 6, 12)
N_SENSORS = 8
VALID_MIN = 1.0
MAX_EPOCHS = 10
PATIENCE = 3
BATCH_SIZE = 256
DEVICE = torch.device('cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(2, os.cpu_count() or 1))
torch.use_deterministic_algorithms(True, warn_only=True)
print('Thiết bị:', DEVICE, '| Seed:', SEED)


In [ ]:
df = pd.read_hdf(DATA, key='df')
assert df.shape == (34272, 207), f'Kích thước dữ liệu không như dự kiến: {df.shape}'
assert isinstance(df.index, pd.DatetimeIndex)
assert (df.index.to_series().diff().iloc[1:] == pd.Timedelta(minutes=5)).all()
sensors = list(map(str, df.columns[:N_SENSORS]))
raw = df.iloc[:, :N_SENSORS].to_numpy(dtype=np.float32)
T, N = raw.shape
train_end, val_end = int(T * .70), int(T * .85)
splits = {'train': (0, train_end), 'validation': (train_end, val_end), 'test': (val_end, T)}

# Imputation nhân quả: chỉ sử dụng quá khứ. Median dự phòng học trên train.
valid_raw = np.isfinite(raw) & (raw > VALID_MIN)
fallback = np.array([np.median(raw[:train_end, j][valid_raw[:train_end, j]]) for j in range(N)], dtype=np.float32)
clean = pd.DataFrame(np.where(valid_raw, raw, np.nan)).ffill().fillna(pd.Series(fallback)).to_numpy(dtype=np.float32)
scaler = StandardScaler().fit(clean[:train_end])
scaled = scaler.transform(clean).astype(np.float32)

# Origin thuộc tập nào thì cả t+3, t+6, t+12 phải nằm trong cùng tập.
def origins_for(split, stride=1):
    start, end = splits[split]
    return np.arange(max(start, LOOKBACK - 1), end - max(HORIZONS), stride, dtype=np.int32)

def make_windows(origins):
    past_idx = origins[:, None] - np.arange(LOOKBACK-1, -1, -1, dtype=np.int32)[None, :]
    target_idx = origins[:, None] + np.array(HORIZONS)[None, :]
    x = scaled[past_idx].astype(np.float32)
    true = raw[target_idx]
    mask = np.isfinite(true) & (true > VALID_MIN)
    y = ((true - scaler.mean_[None,None,:]) / scaler.scale_[None,None,:]).astype(np.float32)
    y = np.where(mask, y, 0).astype(np.float32)
    return x, y, mask.astype(np.float32)

tr_o, va_o, te_o = origins_for('train',2), origins_for('validation',2), origins_for('test')
xtr,ytr,mtr = make_windows(tr_o)
xva,yva,mva = make_windows(va_o)
print('Full data:',df.shape, '| sensors:', sensors)
print('Train:', xtr.shape, ytr.shape, '| val:',xva.shape, yva.shape)
print('Test origins:',len(te_o), '| Horizons:',HORIZONS)
assert xtr.shape[1:] == (LOOKBACK,N) and ytr.shape[1:] == (3,N)
assert len(set(tr_o) & set(va_o)) == 0
assert max(tr_o) + max(HORIZONS) < train_end
assert max(va_o) + max(HORIZONS) < val_end


**Thiết kế đầu vào:** `X ∈ R^(samples, 12, 8)`, `Y ∈ R^(samples, 3, 8)`. Mỗi origin sử dụng đúng 12 mốc đến thời điểm hiện tại; nhãn tại +3, +6 và +12 mốc. Các tập tách theo thời gian; scaler chỉ fit trên train. Giá trị mục tiêu không hợp lệ được mask khỏi hàm loss, không bị xem là nhãn bằng 0.

## 2. Kiến trúc LSTM và huấn luyện

In [ ]:
class TrafficLSTM(nn.Module):
    def __init__(self, n_sensors, horizons):
        super().__init__()
        self.layer1 = nn.LSTM(input_size=n_sensors, hidden_size=64, batch_first=True)
        self.layer2 = nn.LSTM(input_size=64, hidden_size=32, batch_first=True)
        self.dropout = nn.Dropout(0.10)
        self.head = nn.Linear(32, horizons*n_sensors)
        self.horizons, self.n_sensors = horizons, n_sensors
    def forward(self, x):
        z,_ = self.layer1(x)
        z,_ = self.layer2(z)
        return self.head(self.dropout(z[:,-1,:])).reshape(-1,self.horizons,self.n_sensors)

model = TrafficLSTM(N, len(HORIZONS)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_loader = DataLoader(TensorDataset(*map(torch.from_numpy,(xtr,ytr,mtr))),
                          batch_size=BATCH_SIZE, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(TensorDataset(*map(torch.from_numpy,(xva,yva,mva))),
                        batch_size=512, shuffle=False)
print(model)
print('Tham số có thể học:',sum(p.numel() for p in model.parameters()))
print('Output shape:', tuple(model(torch.from_numpy(xtr[:2])).shape))


In [ ]:
def epoch_loss(loader, train=False):
    model.train(train)
    squared_sum, valid_count = 0., 0.
    with torch.set_grad_enabled(train):
        for x, y, mask in loader:
            x,y,mask = x.to(DEVICE),y.to(DEVICE),mask.to(DEVICE)
            pred = model(x)
            squared_error = ((pred-y)**2 * mask).sum()
            count = mask.sum().clamp(min=1.)
            loss = squared_error / count
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step()
            squared_sum += float(squared_error.detach())
            valid_count += float(mask.sum())
    return squared_sum / max(valid_count,1.)

ckpt = OUTPUT / 'lstm_best.pt'
start = time.perf_counter()
best, stale, history = float('inf'),0,[]
for epoch in range(1,MAX_EPOCHS+1):
    train_mse = epoch_loss(train_loader,True)
    val_mse = epoch_loss(val_loader,False)
    history.append({'epoch':epoch,'train_masked_mse':train_mse,'val_masked_mse':val_mse})
    if val_mse < best-1e-5:
        best, stale = val_mse, 0
        torch.save(model.state_dict(),ckpt)
    else:
        stale += 1
    print(f'Epoch {epoch:02d}: train {train_mse:.4f} | val {val_mse:.4f}')
    if stale >= PATIENCE: break
training_seconds = round(time.perf_counter()-start,2)
loss_df = pd.DataFrame(history)
loss_df.to_csv(TABLE/'lstm_loss.csv',index=False)
model.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True))
model.eval()
best_epoch = int(loss_df.loc[loss_df.val_masked_mse.idxmin(),'epoch'])
print('Best epoch:',best_epoch,'| Best val masked MSE:',round(best,4),'| Training seconds:',training_seconds)


## 3. Loss curves và đánh giá dấu hiệu overfitting

In [ ]:
fig,ax = plt.subplots(figsize=(9,4.5))
ax.plot(loss_df.epoch,loss_df.train_masked_mse,'o-',label='Train masked MSE')
ax.plot(loss_df.epoch,loss_df.val_masked_mse,'s-',label='Validation masked MSE')
ax.axvline(best_epoch,linestyle=':',color='gray',label=f'Best epoch = {best_epoch}')
ax.set(xlabel='Epoch',ylabel='Masked MSE (standardized)',title='LSTM · Quá trình học trên METR-LA (8 sensors)')
ax.grid(alpha=.2);ax.legend();fig.tight_layout()
fig.savefig(FIG/'lstm_learning_curve.png',dpi=180)
plt.show()
print('Train MSE cuối:',round(float(loss_df.train_masked_mse.iloc[-1]),4),
      '| Val MSE cuối:',round(float(loss_df.val_masked_mse.iloc[-1]),4))
if best_epoch < len(loss_df):
    print('Validation tốt nhất trước epoch cuối: có dấu hiệu chững hoặc tăng loss; checkpoint ở epoch tốt nhất.')
else:
    print('Validation tốt nhất tại epoch cuối: chưa thấy xu hướng overfitting tăng bền vững trong số epoch đã chạy.')


Loss được tính trong không gian chuẩn hoá và chỉ trên nhãn có mask hợp lệ. Chênh lệch train/validation không tự chứng minh overfitting; cần xem xu hướng validation qua các epoch và sử dụng checkpoint tốt nhất theo validation. Lookback 60 phút chỉ mô tả biến thiên ngắn hạn, chưa trực tiếp mô hình hóa chu kỳ ngày/tuần.

## 4. Dự báo, inverse scaling và đánh giá test

In [ ]:
pred_norm=[]
with torch.no_grad():
    for start_i in range(0,len(te_o),512):
        xx,_,_ = make_windows(te_o[start_i:start_i+512])
        pred_norm.append(model(torch.from_numpy(xx).to(DEVICE)).cpu().numpy())
pred_norm=np.concatenate(pred_norm,axis=0)
# Inverse đúng trục sensor:  mean/scale kích thước (N,) broadcast vào (samples,3,N)
pred = (pred_norm*scaler.scale_[None,None,:]+scaler.mean_[None,None,:]).astype(np.float32)
_,_,test_mask=make_windows(te_o)
idx = te_o[:,None] + np.array(HORIZONS)[None,:]
actual = raw[idx]
assert pred.shape == actual.shape == test_mask.shape == (len(te_o),3,N)
np.save(OUTPUT/'lstm_test_forecast.npy',pred)

records=[]
for hi,h in enumerate(HORIZONS):
    ok = test_mask[:,hi,:].astype(bool)
    truth,predh=actual[:,hi,:][ok],pred[:,hi,:][ok]
    error=predh-truth
    records.append({'model':'LSTM','horizon_steps':h,'horizon_minutes':h*5,
        'n_eval':int(ok.sum()),'RMSE':float(np.sqrt(np.mean(error**2))),
        'MAE':float(np.mean(np.abs(error))),
        'MAPE_pct':float(np.mean(np.abs(error)/np.abs(truth))*100),
        'mask_policy':f'finite and speed > {VALID_MIN}',
        'unit':'METR-LA_raw_speed_unit_unverified'})
metrics=pd.DataFrame(records)
metrics.to_csv(TABLE/'lstm_metrics.csv',index=False)
display(metrics[['horizon_minutes','n_eval','RMSE','MAE','MAPE_pct']].round(3))


In [ ]:
# Bàn giao prediction schema TV6: cùng origin/target/sensor/horizon.
frames=[]
for hi,h in enumerate(HORIZONS):
    origins_t = np.repeat(df.index[te_o].to_numpy(),N)
    targets_t = np.repeat(df.index[te_o+h].to_numpy(),N)
    frames.append(pd.DataFrame({
        'dataset':'METR-LA','model':'LSTM','origin':origins_t,
        'target_timestamp':targets_t,'horizon_steps':h,
        'sensor_id':np.tile(np.array(sensors),len(te_o)),
        'y_true':actual[:,hi,:].reshape(-1),'y_pred':pred[:,hi,:].reshape(-1),
        'unit':'METR-LA_raw_speed_unit_unverified',
        'valid_mask':test_mask[:,hi,:].reshape(-1).astype(bool)}))
predictions=pd.concat(frames,ignore_index=True)
predictions.to_csv(TABLE/'lstm_predictions.csv',index=False)
print('Prediction rows:',len(predictions),'| Shape:',pred.shape)
display(predictions.head(5))


## 5. Trực quan dự báo và sai số

In [ ]:
sensor_idx=0
nplot=min(160,len(te_o))
fig,axes=plt.subplots(3,1,figsize=(12,8),sharex=True)
for hi,h in enumerate(HORIZONS):
    ts=df.index[te_o[:nplot]+h]
    axes[hi].plot(ts,actual[:nplot,hi,sensor_idx],label='Thực tế',lw=1.5)
    axes[hi].plot(ts,pred[:nplot,hi,sensor_idx],label='LSTM',lw=1.4)
    axes[hi].set_ylabel('Tốc độ (đơn vị gốc)')
    axes[hi].set_title(f'Cảm biến {sensors[sensor_idx]} · Horizon {h*5} phút')
    axes[hi].grid(alpha=.2)
axes[0].legend(loc='upper right');fig.autofmt_xdate();fig.tight_layout()
fig.savefig(FIG/'lstm_forecast_examples.png',dpi=180)
plt.show()

fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].bar(metrics.horizon_minutes.astype(str),metrics.RMSE)
axes[0].set(xlabel='Horizon (phút)',ylabel='RMSE',title='Sai số theo horizon')
axes[1].bar(metrics.horizon_minutes.astype(str),metrics.MAE)
axes[1].set(xlabel='Horizon (phút)',ylabel='MAE',title='Sai số tuyệt đối theo horizon')
fig.tight_layout();fig.savefig(FIG/'lstm_error_by_horizon.png',dpi=180);plt.show()


## 6. Cấu hình và kiểm tra kết quả

In [ ]:
config = {
    'dataset':'METR-LA','model':'LSTM(64)->LSTM(32)->Dropout(0.10)->Dense(3*N)',
    'sensor_count':N,'selected_sensors':sensors,'lookback_steps':LOOKBACK,
    'horizons_steps':list(HORIZONS),'full_data_sensors':int(df.shape[1]),
    'split_ratio':[0.7,0.15,0.15],'train_end_index':train_end,
    'validation_end_index':val_end,'seed':SEED,'epochs_max':MAX_EPOCHS,
    'epochs_completed':len(loss_df),'best_epoch':best_epoch,
    'checkpoint':'models/lstm_best.pt','batch_size':BATCH_SIZE,
    'optimizer':'Adam','learning_rate':0.001,'early_stop_patience':PATIENCE,
    'train_val_stride':2,'test_stride':1,'training_seconds':training_seconds,
    'mask_policy':f'finite and target > {VALID_MIN}',
    'imputation':'causal forward fill; leading values train-only median',
    'scaler':'StandardScaler fit only on train',
    'unit':'raw unit (not independently verified)'}
(OUTPUT/'lstm_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
assert len(predictions)==len(te_o)*len(HORIZONS)*N
assert set(metrics.horizon_minutes)=={15,30,60}
assert predictions.groupby('horizon_steps').size().eq(len(te_o)*N).all()
assert (predictions.loc[predictions.valid_mask,'y_true']>VALID_MIN).all()
assert np.isfinite(pred).all()
print('LSTM CHECK PASS ·',len(predictions),'prediction rows ·',len(metrics),'horizons')


### Phạm vi diễn giải

- Cấu hình **64/32 units**, dropout 0,10 và lookback 12 là lựa chọn thí nghiệm, không phải cấu hình tối ưu được chứng minh.
- Đây là **tập con 8/207 cảm biến**, do đó không thể báo cáo như benchmark toàn bộ METR-LA. Cần dùng sensor set và protocol được trưởng nhóm chốt để so sánh chính thức.
- Loss validation được dùng chọn checkpoint; test chỉ dùng đánh giá cuối. Dự báo tại một origin không sử dụng nhãn tương lai làm đầu vào.
- Đầu ra tích hợp: `models/lstm_best.pt`, `models/lstm_config.json`, `models/lstm_test_forecast.npy`, `tables/lstm_loss.csv`, `tables/lstm_predictions.csv`, `tables/lstm_metrics.csv`, cùng các hình trong `figures/`.
